# DocDet Training on Google Colab (Optimised)

This notebook runs every DocDet training phase on a Colab GPU with
throughput-optimised defaults for A100 / H100 / T4 / L4.

## Key Colab-specific optimisations applied here

- **Dependencies are installed inline** (no `pip install -r` pointing at a
  repo-relative file that might not exist yet).
- **Datasets are downloaded per-phase directly to local NVMe SSD** and
  deleted once that phase is done. Avoids overflowing the ~120 GB Colab
  disk (DocSynth300K alone is 119 GB). Checkpoints persist on Drive.
- **GPU is auto-detected** at startup: batch size, `num_workers`, and
  `torch.compile` are set automatically for the assigned hardware.
- **`torch.compile` (PyTorch 2.x)** is enabled on Ampere+ GPUs (A100, H100,
  L4) for a free ~15 % forward-pass speedup.

## Phases

1. Environment setup and GPU auto-detection.
2. Mount Drive, authenticate HuggingFace / Kaggle.
3. Download datasets to Drive, then copy to local SSD.
4. Phase 0 (optional): COCO warmup.
5. Phase 1: DocSynth300K synthetic pretraining.
6. Phase 2: multi-dataset real-document fine-tuning.
7. Phase 3: OmniDocBench evaluation.
8. Phase 4 (conditional): targeted fine-tune.
9. ONNX + INT8 export.

Checkpoints are saved to Drive so progress survives Colab disconnects.

## 1. Environment setup

Install dependencies directly so the notebook is fully self-contained
(no `pip install -r <repo-file>` that would fail before the repo is cloned).

Then clone the repo so `layout_detector.*` imports resolve.

In [ ]:
import subprocess, sys

deps = [
    "torch>=2.2,<3.0",
    "torchvision>=0.17,<1.0",
    "numpy>=1.26",
    "Pillow>=10.0",
    "pyarrow>=14.0",
    "huggingface-hub>=0.23",
    "datasets>=2.19",
    "pycocotools>=2.0.7",
    "onnx>=1.16",
    "onnxruntime>=1.17",
    "tensorboard>=2.15",
    "matplotlib>=3.8",
    "kagglehub>=0.3",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + deps,
)
print("Dependencies installed.")

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Zachary-Woo/VisionPDF.git"
REPO_BRANCH = "main"
WORKDIR = Path("/content/VisionPDF")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(WORKDIR)], check=True)
else:
    subprocess.run(["git", "-C", str(WORKDIR), "pull"], check=True)

os.chdir(WORKDIR)

# Colab's Jupyter kernel does not automatically add cwd to sys.path,
# so `import layout_detector.*` would fail without this.
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

# Also propagate to child processes spawned by `!python -m ...` so the
# phase scripts can resolve `layout_detector` when launched as modules.
os.environ["PYTHONPATH"] = str(WORKDIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("Working dir:", Path.cwd())
print("sys.path[0]:", sys.path[0])

## 1b. GPU auto-detection and optimal defaults

Detects the assigned GPU and sets batch size, `num_workers`, and
whether `torch.compile` should be used. These values are referenced
by every training phase below.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Use Runtime -> Change runtime type -> GPU.")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
GPU_ARCH = torch.cuda.get_device_capability(0)

USE_COMPILE = GPU_ARCH >= (8, 0) and hasattr(torch, "compile")

_name_lower = GPU_NAME.lower()
if "a100" in _name_lower or "h100" in _name_lower:
    BS_PHASE0  = 32
    BS_PHASE1  = 48
    BS_PHASE2  = 40
    NUM_WORKERS = 8
elif "l4" in _name_lower or "4090" in _name_lower:
    BS_PHASE0  = 24
    BS_PHASE1  = 32
    BS_PHASE2  = 28
    NUM_WORKERS = 6
else:
    BS_PHASE0  = 12
    BS_PHASE1  = 16
    BS_PHASE2  = 12
    NUM_WORKERS = 4

print(f"GPU           : {GPU_NAME} ({GPU_MEM_GB:.1f} GB)")
print(f"Compute arch  : sm_{GPU_ARCH[0]}{GPU_ARCH[1]}")
print(f"torch.compile : {'enabled' if USE_COMPILE else 'disabled'}")
print(f"Batch sizes   : phase0={BS_PHASE0}  phase1={BS_PHASE1}  phase2={BS_PHASE2}")
print(f"DataLoader workers: {NUM_WORKERS}")

## 2. Mount Drive + configure persistent output paths

Checkpoints go on Drive so they survive Colab disconnects.
Datasets are downloaded to Drive first (persistent), then copied to
local `/content` SSD for fast I/O during training.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/docdet")
WEIGHTS_DIR = DRIVE_ROOT / "weights"
DATASET_DIR = DRIVE_ROOT / "datasets"
REPORT_DIR = DRIVE_ROOT / "reports"
EXPORT_DIR = DRIVE_ROOT / "exports"

LOCAL_DATA = Path("/content/local_data")

for d in (WEIGHTS_DIR, DATASET_DIR, REPORT_DIR, EXPORT_DIR, LOCAL_DATA):
    d.mkdir(parents=True, exist_ok=True)

print("Drive weights :", WEIGHTS_DIR)
print("Drive datasets:", DATASET_DIR)
print("Local SSD data:", LOCAL_DATA)
print("Reports       :", REPORT_DIR)
print("Exports       :", EXPORT_DIR)

## 3. Authenticate external services

Hugging Face: DocSynth300K and DocLayNet are both on the HF Hub.
Kaggle (optional): required for IIIT-AR-13K via `kaggle datasets download`.

In [ ]:
import os
from google.colab import userdata

# Retrieve the token from Colab Secrets. Change 'HF_TOKEN' if your secret has a different name.
hf_token = userdata.get('HF_TOKEN')

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print("HF login OK")

## 4. Dataset paths + lazy download helpers

Colab VMs only have ~120 GB local disk. DocSynth300K alone is 119 GB,
so downloading every dataset up-front overflows the disk.

Instead each training phase downloads what it needs, then deletes it
when done (checkpoints are already saved to Drive, so no data is lost).

This cell only **declares paths and helpers**. Nothing is downloaded yet.

In [ ]:
import shutil

from layout_detector.docdet.data import download as dld


def _disk_free_gb(path: Path) -> float:
    """Free disk bytes at the given mount point, in GB."""
    total, used, free = shutil.disk_usage(str(path))
    return free / (1024 ** 3)


def drop_dataset(name: str) -> None:
    """
    Delete a local-SSD dataset cache to free disk for the next phase.
    Safe to call even if the folder is already gone.
    """
    target = LOCAL_DATA / name
    if target.exists():
        print(f"Removing {target} to free disk ...")
        shutil.rmtree(str(target), ignore_errors=True)
    print(f"Local SSD free: {_disk_free_gb(LOCAL_DATA):.1f} GB")


print(f"Local SSD free: {_disk_free_gb(LOCAL_DATA):.1f} GB")
print("Nothing downloaded yet. Each phase below handles its own data.")

## 4b. Enable torch.compile on Ampere+ GPUs

Sets the `DOCDET_COMPILE` environment variable that the Trainer reads.
On A100/H100/L4, this gives a ~15% forward-pass speedup for free.
On older GPUs (T4, V100) it's disabled to avoid compilation overhead.

In [ ]:
import os

if USE_COMPILE:
    os.environ["DOCDET_COMPILE"] = "1"
    print("DOCDET_COMPILE=1  (torch.compile will be used by the Trainer)")
else:
    os.environ["DOCDET_COMPILE"] = "0"
    print("DOCDET_COMPILE=0  (torch.compile skipped for this GPU)")

## 5. Phase 0 (optional) - COCO warmup

Only run if you want your head to see "natural" objects before documents. Skip if your goal is pure documents.

In [ ]:
RUN_PHASE0 = False

PHASE0_SAVE_DIR = WEIGHTS_DIR / "phase0"

if RUN_PHASE0:
    PHASE0_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    !python -m layout_detector.docdet.train.phase0_coco \
        --coco-annotations {DATASET_DIR}/coco/annotations/instances_train2017.json \
        --coco-images {DATASET_DIR}/coco/train2017 \
        --save-dir {PHASE0_SAVE_DIR} \
        --batch-size {BS_PHASE0} \
        --num-workers {NUM_WORKERS}

## 6. Phase 1 - DocSynth300K synthetic pretraining

Backbone frozen (first 2 stages). 20 epochs on ~300K synthetic pages.

**Disk usage: ~119 GB while running.** Set `CLEANUP_AFTER_PHASE1 = True`
to auto-delete DocSynth300K once training completes, freeing disk for
Phase 2.

In [ ]:
CLEANUP_AFTER_PHASE1 = True

PHASE1_SAVE_DIR = WEIGHTS_DIR / "phase1"
PHASE1_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# IMPORTANT: pass LOCAL_DATA (the parent cache root), NOT the resolved
# dataset folder. `ensure_docsynth(cache_root=X)` returns X/docsynth, so
# passing that back in would create X/docsynth/docsynth and trigger a
# second full download.
print(f"Disk free before DocSynth download: {_disk_free_gb(LOCAL_DATA):.1f} GB")
DOCSYNTH_DIR = dld.ensure_docsynth(cache_root=LOCAL_DATA)
print(f"Disk free after DocSynth download : {_disk_free_gb(LOCAL_DATA):.1f} GB")
print(f"DocSynth resolved at: {DOCSYNTH_DIR}")

PHASE0_CKPT = PHASE0_SAVE_DIR / "docdet_phase0_best.pt"
phase0_arg = f"--coco-init-from {PHASE0_CKPT}" if PHASE0_CKPT.exists() else ""

!python -m layout_detector.docdet.train.phase1_synth \
    --cache-root {LOCAL_DATA} \
    --save-dir {PHASE1_SAVE_DIR} \
    --batch-size {BS_PHASE1} \
    --num-workers {NUM_WORKERS} \
    {phase0_arg}

if CLEANUP_AFTER_PHASE1:
    drop_dataset("docsynth")

## 7. Phase 2 - real-document multi-dataset fine-tune

Backbone unfrozen. Weighted sampler mixes the four real-document datasets:

| Source     | Format     | Size on disk | Mix weight | Toggle              |
|------------|------------|--------------|------------|---------------------|
| DocLayNet  | parquet    | ~40 GB       | 3.0        | always on           |
| PubLayNet  | parquet    | ~25 GB train | 1.0        | `USE_PUBLAYNET`     |
| TableBank  | parquet    | ~15 GB       | 2.0        | `USE_TABLEBANK`     |
| IIIT-AR-13K| Pascal VOC | ~1.5 GB      | 4.0        | `USE_IIIT_AR`       |

Each dataset is downloaded into `LOCAL_DATA` only when needed and deleted
afterwards if its `CLEANUP_*` flag is set, keeping the Colab disk small.

IIIT-AR-13K is hosted on Kaggle. The next cell uses `kagglehub` which
will prompt once for your Kaggle username + API key (or read
`~/.kaggle/kaggle.json`).

In [ ]:
import os, getpass

USE_IIIT_AR = True

if USE_IIIT_AR:
    if not os.environ.get("KAGGLE_USERNAME"):
        os.environ["KAGGLE_USERNAME"] = getpass.getpass("Kaggle username: ")
    if not os.environ.get("KAGGLE_KEY"):
        os.environ["KAGGLE_KEY"] = getpass.getpass("Kaggle API key: ")
    print("Kaggle credentials set for kagglehub.")
else:
    print("IIIT-AR-13K disabled; skipping Kaggle credential prompt.")

In [ ]:
USE_PUBLAYNET = True
USE_TABLEBANK = True
# USE_IIIT_AR was set in the previous cell

CLEANUP_PUBLAYNET_AFTER_PHASE2 = True
CLEANUP_TABLEBANK_AFTER_PHASE2 = True
CLEANUP_IIIT_AR_AFTER_PHASE2  = False  # tiny dataset, kept for Phase 4

PHASE2_SAVE_DIR = WEIGHTS_DIR / "phase2"
PHASE2_SAVE_DIR.mkdir(parents=True, exist_ok=True)
PHASE1_CKPT = PHASE1_SAVE_DIR / "docdet_phase1_synth_final.pt"

print(f"Disk free before Phase 2 downloads: {_disk_free_gb(LOCAL_DATA):.1f} GB")

DOCLAYNET_DIR = dld.ensure_doclaynet(cache_root=LOCAL_DATA)
extra_args = [f"--doclaynet-root {DOCLAYNET_DIR}"]

if USE_PUBLAYNET:
    PUBLAYNET_DIR = dld.ensure_publaynet(cache_root=LOCAL_DATA)
    extra_args.append(f"--publaynet-root {PUBLAYNET_DIR}")

if USE_TABLEBANK:
    TABLEBANK_DIR = dld.ensure_tablebank(cache_root=LOCAL_DATA)
    extra_args.append(f"--tablebank-root {TABLEBANK_DIR}")

if USE_IIIT_AR:
    IIIT_AR_DIR = dld.ensure_iiit_ar(cache_root=LOCAL_DATA)
    extra_args.append(f"--iiit-ar-root {IIIT_AR_DIR}")

print(f"Disk free after Phase 2 downloads : {_disk_free_gb(LOCAL_DATA):.1f} GB")

extra_args_str = " ".join(extra_args)
!python -m layout_detector.docdet.train.phase2_real \
    --phase1-checkpoint {PHASE1_CKPT} \
    {extra_args_str} \
    --save-dir {PHASE2_SAVE_DIR} \
    --batch-size {BS_PHASE2} \
    --num-workers {NUM_WORKERS}

if USE_PUBLAYNET and CLEANUP_PUBLAYNET_AFTER_PHASE2:
    drop_dataset("publaynet")
if USE_TABLEBANK and CLEANUP_TABLEBANK_AFTER_PHASE2:
    drop_dataset("tablebank")
if USE_IIIT_AR and CLEANUP_IIIT_AR_AFTER_PHASE2:
    drop_dataset("iiit_ar")

## 8. Phase 3 - evaluation on OmniDocBench

OmniDocBench is held-out. This produces `phase3_report.json` with mAP / per-class AP / Table AP@0.5. Spec thresholds: overall mAP@0.5:0.95 >= 0.70, Table AP@0.5 >= 0.90.

In [ ]:
PHASE2_CKPT = PHASE2_SAVE_DIR / "docdet_phase2_real_final.pt"
PHASE3_REPORT_DIR = REPORT_DIR / "phase3"

OMNIDOCBENCH_DIR = dld.ensure_omnidocbench(cache_root=LOCAL_DATA)

!python -m layout_detector.docdet.eval.eval_omnidocbench \
    --checkpoint {PHASE2_CKPT} \
    --annotations {OMNIDOCBENCH_DIR}/OmniDocBench.json \
    --image-root {OMNIDOCBENCH_DIR}/images \
    --output-dir {PHASE3_REPORT_DIR}

import json
report_file = next(PHASE3_REPORT_DIR.glob("*.json"))
with open(report_file) as f:
    report = json.load(f)
print(json.dumps(report, indent=2))

## 9. Phase 4 (conditional) - targeted fine-tune

Trigger Phase 4 only if Phase 3 showed a specific class underperforming (e.g. Formula AP < 0.50 or Table AP@0.5 < 0.90). `--boost-class` can be repeated.

In [ ]:
BOOST_CLASSES = []  # e.g. ["Formula", "Table"]

if BOOST_CLASSES:
    PHASE4_SAVE_DIR = WEIGHTS_DIR / "phase4"
    PHASE4_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    boost_args = " ".join(f"--boost-class {c}" for c in BOOST_CLASSES)

    !python -m layout_detector.docdet.train.phase4_targeted \
        --phase2-checkpoint {PHASE2_CKPT} \
        --doclaynet-root {DOCLAYNET_DIR} \
        --save-dir {PHASE4_SAVE_DIR} \
        --batch-size {BS_PHASE2} \
        --num-workers {NUM_WORKERS} \
        --boost-factor 2.0 \
        {boost_args}
    FINAL_CKPT = PHASE4_SAVE_DIR / "docdet_phase4_targeted_final.pt"
else:
    FINAL_CKPT = PHASE2_CKPT

print("Final checkpoint:", FINAL_CKPT)

## 10. Export - ONNX + INT8

Export the final PyTorch checkpoint to ONNX (opset 17, dynamic batch/height/width), then calibrate INT8 on 500 DocLayNet validation pages.

In [ ]:
ONNX_FP32 = EXPORT_DIR / "docdet.onnx"
ONNX_INT8 = EXPORT_DIR / "docdet_int8.onnx"
CAL_DIR = DOCLAYNET_DIR / "PNG"

!python -m layout_detector.docdet.export.export_onnx \
    --checkpoint {FINAL_CKPT} \
    --output {ONNX_FP32} \
    --opset 17

!python -m layout_detector.docdet.export.quantize_int8 \
    --fp32-onnx {ONNX_FP32} \
    --int8-onnx {ONNX_INT8} \
    --calibration-images {CAL_DIR} \
    --num-calibration-images 500

## 11. Smoke-test the exported models

Quickly visualise detections on a sample page to make sure the exported ONNX / INT8 models still behave.

In [ ]:
from PIL import Image
from layout_detector.docdet.infer.predict import DocDetPredictor

sample_image = next(CAL_DIR.glob("*.png"))
img = Image.open(sample_image).convert("RGB")

predictor = DocDetPredictor(model_path=ONNX_INT8, device="cpu", score_threshold=0.3)
dets = predictor.predict(img)
for d in dets[:20]:
    print(f"{d.label:<15} score={d.score:.2f} bbox={d.bbox}")

## 12. Training summary + disk cleanup

Print a summary of what was trained, where checkpoints live, and
optionally free the local SSD copy (Drive copy is kept).

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"GPU used        : {GPU_NAME}")
print(f"torch.compile   : {'yes' if USE_COMPILE else 'no'}")
print(f"Final checkpoint: {FINAL_CKPT}")
print(f"ONNX FP32       : {ONNX_FP32}")
print(f"ONNX INT8       : {ONNX_INT8}")
print()

FREE_ALL_LOCAL = False
if FREE_ALL_LOCAL:
    for name in ("docsynth", "doclaynet", "publaynet", "omnidocbench",
                 "docbank", "tablebank", "iiit_ar"):
        drop_dataset(name)
else:
    print(f"Local SSD free: {_disk_free_gb(LOCAL_DATA):.1f} GB")
    print(f"Local datasets kept at {LOCAL_DATA}")
    print("Set FREE_ALL_LOCAL = True to wipe them (checkpoints on Drive are safe).")